In [1]:
import pymongo
from pymongo import MongoClient
from bson.objectid import ObjectId

# في كزا package بستخدمها علشان ابني ال API
from flask import Flask , request ,jsonify
import threading
import time

# pip install pymongo flask

In [2]:
app = Flask(__name__)
# Create a Flask application instance
# This object represents your API server
# It handles requests and responses

client = MongoClient("mongodb://localhost:27017/")
# Create a connection to MongoDB
# localhost means the database is running on the same machine
# 27017 is the default MongoDB port

db = client['Retail_Business']
# Select the database named Retail_Business
# If it does not exist MongoDB will create it automatically

customers_col = db['customers_warehouse']
# Select the collection named customers
# A collection is similar to a table in SQL
# This is where customer data will be stored


In [3]:
@app.route("/health")
def health():
    return jsonify({"status": "ok"})

@app.route('/top-customers', methods=['GET'])
def top_customers():
    result = list(
        customers_col
        .find({}, {"_id": 0})
        .sort("total_spent", -1)
        .limit(5)
    )
    return jsonify(result)

@app.route('/best-products', methods=['GET'])
def best_products():
    pipeline = [
        {
            "$group": {
                "_id": "$product.name",
                "total_quantity": { "$sum": "$product.quantity" }
            }
        },
        {
            "$sort": { "total_quantity": -1 }
        },
        {
            "$limit": 5
        }
    ]

    result = list(db.Orders.aggregate(pipeline))

    # تجهيز الشكل للـ JSON
    for doc in result:
        doc["product_name"] = doc["_id"]
        del doc["_id"]

    return jsonify(result)


@app.route('/branch-revenue', methods=['GET'])
def branch_revenue():
    pipeline = [
        {
            "$group": {
                "_id": "$branch.name",
                "total_revenue": { "$sum": "$sale.total_amount" }
            }
        },
        {
            "$sort": { "total_revenue": -1 }
        }
    ]

    result = list(db.Orders.aggregate(pipeline))

    # تجهيز النتيجة للـ JSON
    for doc in result:
        doc["branch_name"] = doc["_id"]
        del doc["_id"]

    return jsonify(result)


@app.route('/monthly-sales', methods=['GET'])
def monthly_sales():
    pipeline = [
        {
            "$project": {
                "year_month": {
                    "$substr": ["$sale.date", 0, 7]
                },
                "amount": "$sale.total_amount"
            }
        },
        {
            "$group": {
                "_id": "$year_month",
                "total_sales": { "$sum": "$amount" }
            }
        },
        {
            "$sort": { "_id": 1 }
        }
    ]

    result = list(db.Orders.aggregate(pipeline))

    for doc in result:
        doc["month"] = doc["_id"]
        del doc["_id"]

    return jsonify(result)


@app.route('/best-products-by-branch', methods=['GET'])
def best_products_by_branch():
    result = list(
        db.products_branch_warehouse.find({}, {"_id": 0})
    )
    return jsonify(result)



@app.route('/seasonal-product-demand', methods=['GET'])
def seasonal_product_demand():
    result = list(
        db.seasonal_product_warehouse.find({}, {"_id": 0})
    )
    return jsonify(result)


@app.route('/stock-planning', methods=['GET'])
def stock_planning():
    result = list(
        db.stock_planning_warehouse.find({}, {"_id": 0})
    )
    return jsonify(result)





In [4]:
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_flask).start()
time.sleep(1)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Feb/2026 00:21:00] "GET /top-customers HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:00] "GET /best-products HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:01] "GET /best-products-by-branch HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:02] "GET /branch-revenue HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:02] "GET /monthly-sales HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:03] "GET /seasonal-product-demand HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:04] "GET /stock-planning HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:06] "GET /top-customers HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:06] "GET /best-products HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:07] "GET /best-products-by-branch HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:08] "GET /branch-revenue HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:08] "GET /monthly-sales HTTP/1.1" 200 -
127.0.0.1 - - [05/Feb/2026 00:21:1